# Pointing, potentiometer, IMU and a transmitter tooth through the elevation-sweep raster

The beam-mapping raster of 07-17: continuous ±180° elevation sweeps from
**20:24 to 21:28 UTC** while the azimuth advances slowly from ~190° through
360° to ~52°. The transmitter comb is on throughout, and pointing quality is
`ok` for the whole raster.

Plotted against time: solved pointing (`az_deg`, `el_deg`), the raw
potentiometer azimuth (`pot_az_deg`) and the IMU elevation (`imu_el_deg`) from
`curation/pointing_table.parquet`, and `box-air` power on one transmitter
comb tooth next to the mid-gap channel four channels away. The mid-gap channel
is the sky and terrain background under the tooth. Section 1 is the whole
raster. Section 2 zooms in on a few sweeps at three different azimuths.

**How the raster was found**, since earlier versions of this notebook got it
wrong. The pointing table was plotted over the full documented beam-scan
window, 07-17 18:48:52 → 07-18 03:08:49 UTC (`pointing/README.md`). Two
earlier mistakes are corrected here:
- A search for sweeps "at fixed azimuth" required the azimuth to hold within 8°
  for 10 minutes. The raster's azimuth advances ~4°/min, so it was rejected,
  and the stepped moves at 19:50–20:15 were shown instead.
- The beam-scan time constant used elsewhere started and ended two hours late.

**The transmitter is the 8-channel comb** (1.953125 MHz spacing, channel index
≡ 0 mod 8). The archive calls it the "digital self-comb", and
`tx_presence.jsonl` reports the transmitter off. Both are wrong.
`self_comb_presence.jsonl` is still the right per-file record of when the comb
was on.

In [ ]:
from pathlib import Path
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib widget
import pyarrow.parquet as pq

PLOT_BACKEND = os.environ.get('EIGSEP_NOTEBOOK_BACKEND', 'widget')
get_ipython().run_line_magic('matplotlib', PLOT_BACKEND)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'marjum-2026-07/data').is_dir())
CAMPAIGN = ROOT / 'marjum-2026-07'

import eigsep_data
from eigsep_data import AntennaResolutionPolicy, MetadataIndex
eigsep_data.set_campaign_root(CAMPAIGN)

WINDOW_UTC = ('2026-07-17 20:22', '2026-07-17 21:30')     # the whole raster
ZOOMS_UTC = [('2026-07-17 20:26:30', '2026-07-17 20:30:30'),   # az ~ 188-200
             ('2026-07-17 20:46:00', '2026-07-17 20:50:00'),   # az ~ 258-265
             ('2026-07-17 21:16:00', '2026-07-17 21:20:00')]   # az ~ 15-30
# A transmitter comb tooth (index % 8 == 0). 170-200 MHz is where the tooth
# stands 200-400x above the gap; at 70 MHz it is within a few percent.
TOOTH_MHZ = 189.4531
GAP_OFFSET_CH = 4
ANTENNA = 'box-air'

In [ ]:
t0, t1 = (pd.Timestamp(s, tz='UTC').timestamp() for s in WINDOW_UTC)

pointing = pq.read_table(CAMPAIGN / 'curation/pointing_table.parquet').to_pandas()
pointing = pointing[(pointing.t_utc_s >= t0) & (pointing.t_utc_s <= t1)].sort_values('t_utc_s')
pointing['time'] = pd.to_datetime(pointing.t_utc_s, unit='s', utc=True)
ok = pointing.quality == 'ok'
print('pointing rows %d; quality: %s' % (len(pointing), pointing.quality.value_counts().to_dict()))

policy = AntennaResolutionPolicy.load(CAMPAIGN / 'curation/antenna_resolution.json')
bundle = MetadataIndex(CAMPAIGN / 'data').select(time=(t0, t1)).load_bundle(
    antenna=ANTENNA, missing='skip', resolution_policy=policy)
CH_WIDTH = 250.0 / 1024
tooth = int(np.argmin(np.abs(bundle.freqs_mhz - TOOTH_MHZ)))
gap = tooth + GAP_OFFSET_CH
tooth_index = int(round(bundle.freqs_mhz[tooth] / CH_WIDTH))
if tooth_index % 8:
    print('warning: channel %d is not on the 8-channel comb' % tooth_index)
sky = (bundle.meta.rfswitch == 'RFANT').to_numpy()
power_time = pd.to_datetime(bundle.meta.time_best.astype(float), unit='s', utc=True)
p_tooth, p_gap = bundle.data[:, tooth], bundle.data[:, gap]

comb = {}
for line in open(CAMPAIGN / 'curation/self_comb_presence.jsonl'):
    row = json.loads(line)
    if 'file' in row:
        comb[row['file']] = bool(row.get('self_comb_on'))
comb_on = np.array([comb.get(f, False) for f in bundle.meta.file])
print('%s tooth ch %d = %.4f MHz, gap ch %d = %.4f MHz; %d RFANT rows of %d; '
      'comb on in %d of %d files'
      % (ANTENNA, tooth_index, bundle.freqs_mhz[tooth], tooth_index + GAP_OFFSET_CH,
         bundle.freqs_mhz[gap], sky.sum(), sky.size,
         sum(comb.get(f, False) for f in set(bundle.meta.file)), bundle.meta.file.nunique()))


def wrap180(deg):
    # Azimuth on (-180, 180], so the pass through 0/360 does not jump the panel.
    return (np.asarray(deg, float) + 180.0) % 360.0 - 180.0

In [ ]:
plt.rcParams.update({'font.size': 14, 'axes.spines.top': False,
                     'axes.spines.right': False})


def plot_window(start, stop, title):
    a, b = (pd.Timestamp(s, tz='UTC') for s in (start, stop))
    pm = (pointing.time >= a) & (pointing.time <= b)
    ok = pm & (pointing.quality == 'ok')
    dm = (power_time >= a) & (power_time <= b) & sky

    fig, axes = plt.subplots(3, 1, figsize=(15, 11), sharex=True, constrained_layout=True)
    ax = axes[0]
    ax.plot(pointing.time[pm], wrap180(pointing.pot_az_deg[pm]), '.', ms=4, color='0.75',
            label='potentiometer')
    ax.plot(pointing.time[ok], wrap180(pointing.az_deg[ok]), '.', ms=2, color='tab:blue',
            label='solved azimuth')
    ax.set_ylabel('Azimuth [deg]')
    ax.legend(loc='best', markerscale=4)

    ax = axes[1]
    ax.plot(pointing.time[pm], pointing.imu_el_deg[pm], '.', ms=4, color='0.75',
            label='IMU elevation')
    ax.plot(pointing.time[ok], pointing.el_deg[ok], '.', ms=2, color='tab:blue',
            label='solved elevation')
    ax.set_ylabel('Elevation [deg]')
    ax.legend(loc='best', markerscale=4)

    ax = axes[2]
    ax.plot(power_time[dm], p_gap[dm], '.', ms=3, color='0.6',
            label='gap %.3f MHz (background)' % bundle.freqs_mhz[gap])
    ax.plot(power_time[dm], p_tooth[dm], '.', ms=3, color='tab:red',
            label='tooth %.3f MHz (transmitter + background)' % bundle.freqs_mhz[tooth])
    off = dm & ~comb_on
    if off.any():
        ax.fill_between(power_time, 0, 1, where=(~comb_on), transform=ax.get_xaxis_transform(),
                        color='0.9', zorder=0, label='comb off')
    ax.set_yscale('log')
    ax.set_ylabel('%s power [counts]' % ANTENNA)
    ax.set_xlabel('Time [UTC]')
    ax.legend(loc='lower right', markerscale=4, frameon=True, framealpha=0.9)
    axes[0].set_title(title)
    fig.autofmt_xdate()
    plt.show()

## 1. The whole raster

In [ ]:
plot_window(*WINDOW_UTC, 'Elevation-sweep raster, %s to %s UTC' % WINDOW_UTC)

## 2. A few sweeps at three azimuths

Each zoom is four minutes long, covering a few full elevation sweeps while the
azimuth barely moves.

In [ ]:
for start, stop in ZOOMS_UTC:
    a, b = (pd.Timestamp(s, tz='UTC') for s in (start, stop))
    m = (pointing.time >= a) & (pointing.time <= b) & (pointing.quality == 'ok')
    az_range = wrap180(pointing.az_deg[m])
    plot_window(start, stop, '%s–%s UTC,  azimuth %.0f° to %.0f°'
                % (start[-8:], stop[-8:], az_range.min() % 360, az_range.max() % 360))

## One spectrum at az = 0, el = 0

The single integration in the window whose solved pointing is nearest
(az, el) = (0, 0). Only `quality == 'ok'` samples count, and azimuth is compared
across the 0/360 wrap. The `box-air` and `box-gnd` autos and the magnitude of
their cross are taken from that same integration. The policy resolves the cross
to its key per file. The transmitter comb shows as the regular spikes every
1.95 MHz from ~55 MHz upward. On the teeth the cross is as strong as the autos,
as it should be for a single source both inputs see. Between the teeth the cross
sits 1–2 decades below the autos.

In [ ]:
TARGET_AZ, TARGET_EL = 0.0, 0.0

cand = pointing[ok].copy()
d_az = np.abs(wrap180(cand.az_deg - TARGET_AZ))
cand['dist'] = np.hypot(d_az, cand.el_deg - TARGET_EL)
best = cand.loc[cand.dist.idxmin()]
print('nearest ok sample: %s  az %.2f  el %.2f  (%.2f deg from target)  file %s'
      % (best.time.strftime('%H:%M:%S'), best.az_deg % 360, best.el_deg, best.dist, best.file))

# Load just that file for each product and take the row nearest in time.
one = MetadataIndex(CAMPAIGN / 'data').select(files=(best.file,))
spectra = {}
for name in ('box-air', 'box-gnd', 'cross'):
    b = one.load_bundle(antenna=name, missing='skip', resolution_policy=policy)
    row = int(np.argmin(np.abs(b.meta.time_best.astype(float).to_numpy() - best.t_utc_s)))
    spectra[name] = (b.freqs_mhz, np.abs(b.data[row]), float(b.meta.time_best.iloc[row]),
                     b.meta.rfswitch.iloc[row])
for name, (_, _, t_row, sw) in spectra.items():
    print('  %-8s row at %+.2f s from the pointing sample, rfswitch %s'
          % (name, t_row - best.t_utc_s, sw))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6.5), constrained_layout=True)
for name, color in (('box-air', 'tab:blue'), ('box-gnd', 'tab:orange'), ('cross', 'tab:green')):
    f, amp, _, _ = spectra[name]
    label = '|box-gnd × box-air|' if name == 'cross' else name
    ax.semilogy(f, amp, color=color, lw=1.2, label=label)
ax.set_xlabel('Frequency [MHz]')
ax.set_ylabel('Amplitude [counts]')
ax.set_title('Single integration at az %.1f°, el %.1f° — %s UTC'
             % (best.az_deg % 360, best.el_deg, best.time.strftime('%Y-%m-%d %H:%M:%S')))
ax.legend(loc='lower center')
plt.show()